# **Test OpenRouter Provider Selection**

## **Load libraries and setup**

In [168]:
from openai import OpenAI

from mdner_llm.common import load_api_key

In [169]:
%load_ext watermark
%watermark

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Last updated: 2026-06-10T18:48:22.172058+02:00

Python implementation: CPython
Python version       : 3.13.13
IPython version      : 8.13.2

Compiler    : Clang 22.1.3 
OS          : Darwin
Release     : 25.4.0
Machine     : arm64
Processor   : arm
CPU cores   : 15
Architecture: 64bit



In [170]:
api_key = load_api_key("OPENROUTER_API_KEY")
# Instantiate an OpenAI client for the requested model
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [171]:
# Define arguments for the API call
kwargs = {
    "model": "google/gemma-4-31b-it",
    "messages": [
        {"role": "system", "content": "Extract entities as structured JSON."},
        {
            "role": "user",
            "content": "Extract entities from the following text:"
            "'Apple is looking at buying U.K. startup for $1 billion'.",
        },
    ],
}

In [ ]:
# Specify provider selection criteria
provider = "wandb/bf16"
if provider:
    kwargs["extra_body"] = {
        "provider": {"order": [provider], "allow_fallbacks": False}
    }

In [173]:
kwargs


{'model': 'google/gemma-4-31b-it',
 'messages': [{'role': 'system',
   'content': 'Extract entities as structured JSON.'},
  {'role': 'user',
   'content': "Extract entities from the following text:'Apple is looking at buying U.K. startup for $1 billion'."}],
 'extra_body': {'provider': {'order': ['wandb/bf16'],
   'allow_fallbacks': True}}}

In [174]:
# Query the LLM
llm_response = client.chat.completions.create(**kwargs)
llm_response

ChatCompletion(id='gen-1781110102-dHiGWviD7Bmx4tnO1MH6', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```json\n{\n  "entities": [\n    {\n      "text": "Apple",\n      "type": "Organization"\n    },\n    {\n      "text": "U.K.",\n      "type": "Location"\n    },\n    {\n      "text": "$1 billion",\n      "type": "Money"\n    }\n  ]\n}\n```', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='stop')], created=1781110102, model='google/gemma-4-31b-it-20260402', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=87, prompt_tokens=58, total_tokens=145, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0,

```json
ChatCompletion(
    id="gen-1781090496-X8pVkleggEgzVrDmPeDI",
    choices=[
        Choice(
            finish_reason="stop",
            index=0,
            logprobs=None,
            message=ChatCompletionMessage(
                content='```json\n{\n  "entities": [\n    {\n      "name": "Apple",\n      "type": "Organization"\n    },\n    {\n      "name": "U.K. startup",\n      "type": "Organization"\n    },\n    {\n      "value": "$1 billion",\n      "type": "Money"\n    }\n  ]\n}\n```',
                refusal=None,
                role="assistant",
                annotations=None,
                audio=None,
                function_call=None,
                tool_calls=None,
                reasoning=None,
            ),
            native_finish_reason="stop",
        )
    ],
    created=1781090496,
    model="openai/gpt-4o",
    object="chat.completion",
    service_tier=None,
    system_fingerprint="fp_2c1bd5cc0e",
    usage=CompletionUsage(
        completion_tokens=71,
        prompt_tokens=38,
        total_tokens=109,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=None,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=None,
            image_tokens=0,
        ),
        prompt_tokens_details=PromptTokensDetails(
            audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0
        ),
        cost=0,
        is_byok=True,
        cost_details={
            "upstream_inference_cost": 0.000805,
            "upstream_inference_prompt_cost": 9.5e-05,
            "upstream_inference_completions_cost": 0.00071,
        },
    ),
    provider="OpenAI",
)
```

In [175]:
# Display the only information about the provider used for the response
llm_response.provider


'WandB'

In [176]:
# Display the content of the response
llm_response.choices[0].message.content

'```json\n{\n  "entities": [\n    {\n      "text": "Apple",\n      "type": "Organization"\n    },\n    {\n      "text": "U.K.",\n      "type": "Location"\n    },\n    {\n      "text": "$1 billion",\n      "type": "Money"\n    }\n  ]\n}\n```'